In [1]:
import numpy as np
import pandas as pd
import h5py
import os
import time
import seaborn as sns
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import write_to_hdf5_file_single_array, read_in_routine, write_to_hdf5_file
from sleep_analysis_functions import compute_sleep_indices
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import itertools
import pickle

from icu_sleep_breathing_2020_help_functions import * 

import matplotlib


font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 6}

font_headers =  {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 8}

font_subplots =  {'family' : 'sans-serif',
        'weight' : 'bold',
        'size'   : 6}

matplotlib.rc('font', **font)
matplotlib.rc('font', **font)

In [2]:
plots_savedir = '/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Plots/discordance_agreement_with_experts'
if not os.path.exists(plots_savedir):
    os.makedirs(plots_savedir)

In [3]:
data = pickle.load( open("/scr/wolfgang/Sleep_And_Breathing/sleep_staging_lhl/data_w_pca_umap.p", "rb" ) )


In [4]:
# [x for x in data.columns if 'h_' not in x] 'stage_pred_ecg_nn',
cols = [ 
'study_id',
'population',
'stage_pred_amendsumeffort',
'stage_pred_ecg_nn',
'stage_pred_expert',
'ecg_nn_amendsumeffort_agreement_relaxed'
       ]

In [5]:
data[cols]

,study_id,population,stage_pred_amendsumeffort,stage_pred_ecg_nn,stage_pred_expert,ecg_nn_amendsumeffort_agreement_relaxed
0,6,sleeplab,3.0,5.0,5.0,True
1,6,sleeplab,3.0,5.0,5.0,True
2,6,sleeplab,3.0,5.0,5.0,True
3,6,sleeplab,3.0,5.0,5.0,True
4,6,sleeplab,3.0,5.0,3.0,True
...,...,...,...,...,...,...
543189,4,icu,5.0,5.0,NaN,True
543190,4,icu,5.0,5.0,NaN,True
543191,4,icu,5.0,5.0,NaN,True
543192,4,icu,5.0,5.0,NaN,True


In [6]:
data = data.loc[data.population == 'sleeplab', cols]
data = data.loc[data.stage_pred_expert != -1]

In [7]:
data.study_id.unique().shape

(220,)

In [8]:
data_concordant = data.query("ecg_nn_amendsumeffort_agreement_relaxed == True").copy()
data_discordant = data.query("ecg_nn_amendsumeffort_agreement_relaxed == False").copy()
assert data_concordant.shape[0] + data_discordant.shape[0] == data.shape[0]

In [9]:
print(data_concordant.shape)
print(data.shape)
print(data_concordant.shape[0] / data.shape[0])

(131157, 6)
(173977, 6)
0.7538755122803589


In [10]:
data_concordant

,study_id,population,stage_pred_amendsumeffort,stage_pred_ecg_nn,stage_pred_expert,ecg_nn_amendsumeffort_agreement_relaxed
0,6,sleeplab,3.0,5.0,5.0,True
1,6,sleeplab,3.0,5.0,5.0,True
2,6,sleeplab,3.0,5.0,5.0,True
3,6,sleeplab,3.0,5.0,5.0,True
4,6,sleeplab,3.0,5.0,3.0,True
...,...,...,...,...,...,...
174668,465,sleeplab,3.0,5.0,3.0,True
174669,465,sleeplab,3.0,5.0,3.0,True
174670,465,sleeplab,3.0,5.0,3.0,True
174671,465,sleeplab,3.0,5.0,3.0,True


In [11]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, cohen_kappa_score
from matplotlib.colors import LinearSegmentedColormap


In [12]:
labels_order = [5, 4, 3, 2, 1]
labels_names = ['W', 'R', 'N1', 'N2', 'N3']

models = ['stage_pred_amendsumeffort', 'stage_pred_ecg_nn']

cm_dict = {}

for model in models:

    print(model)
    
    cm_absolute = confusion_matrix(data_concordant['stage_pred_expert'], data_concordant[model], labels = labels_order)
    cm_normalized = confusion_matrix(data_concordant['stage_pred_expert'], data_concordant[model], labels = labels_order, normalize='true')
    kappa = cohen_kappa_score(data_concordant['stage_pred_expert'], data_concordant[model])
    
    cm_dict[f"concordant_{model}_absolute"] = cm_absolute
    cm_dict[f"concordant_{model}_normalized"] = cm_normalized
    cm_dict[f"concordant_{model}_kappa"] = np.round(kappa, 2)

    cm_absolute = confusion_matrix(data_discordant['stage_pred_expert'], data_discordant[model], labels = labels_order)
    cm_normalized = confusion_matrix(data_discordant['stage_pred_expert'], data_discordant[model], labels = labels_order, normalize='true')
    kappa = cohen_kappa_score(data_discordant['stage_pred_expert'], data_discordant[model])

    cm_dict[f"discordant_{model}_absolute"] = cm_absolute
    cm_dict[f"discordant_{model}_normalized"] = cm_normalized
    cm_dict[f"discordant_{model}_kappa"] = np.round(kappa, 2)

    
# print()
# print(confusion_matrix(data_concordant['stage_pred_expert'], data_concordant[model], normalize='true').round(2))

stage_pred_amendsumeffort
stage_pred_ecg_nn


In [13]:
cmap_ss5_absolute = LinearSegmentedColormap.from_list('ss5_absolute', ['white', 'black'], N=100)
model_labels = {'stage_pred_amendsumeffort': 'Breathing-based model',
               'stage_pred_ecg_nn': 'HRV-based model'}

In [14]:
plt.close('all')

fig, ax = plt.subplots(2, 4, figsize=(7.6, 5))

for i_model, model in enumerate(models):

    # CONCORDANT 
    i_axis = (0, 0 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"concordant_{model}_absolute"], display_labels=labels_names)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='d')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title(f"Confusion Matrix")
    ax[i_axis].set_ylabel(None)

    i_axis = (0, 1 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"concordant_{model}_normalized"], display_labels=labels_names)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis])
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('Confusion Matrix Normalized')
    ax[i_axis].set_ylabel(None)

    # DISCORDANT 
    i_axis = (1, 0 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"discordant_{model}_absolute"], display_labels=labels_names)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='d')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('Confusion Matrix')
    ax[i_axis].set_ylabel(None)

    i_axis = (1, 1 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"discordant_{model}_normalized"], display_labels=labels_names)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis])
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('Confusion Matrix Normalized')
    ax[i_axis].set_ylabel(None)

    
ax[(0, 0)].set_ylabel(f'Expert label')
ax[(1, 0)].set_ylabel(f'Expert label')
plt.text(0.31, 0.97, 'Breathing', transform=fig.transFigure, ha='center', fontsize=8, fontweight='bold')
plt.text(0.77, 0.97, 'HRV', transform=fig.transFigure, ha='center', fontsize=8, fontweight='bold')
plt.text(-0.4, 0.5, 'Concordant data', transform=ax[(0, 0)].transAxes, rotation=90, va='center', fontsize=8, fontweight='bold')
plt.text(-0.4, 0.5, 'Discordant data', transform=ax[(1, 0)].transAxes, rotation=90, va='center', fontsize=8, fontweight='bold')

# add Cohen's kappa
plt.text(0.31, 0.93, f"Cohen's kappa = {cm_dict[f'concordant_{models[0]}_kappa']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.77, 0.93, f"Cohen's kappa = {cm_dict[f'concordant_{models[1]}_kappa']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.31, 0.45, f"Cohen's kappa = {cm_dict[f'discordant_{models[0]}_kappa']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.77, 0.45, f"Cohen's kappa = {cm_dict[f'discordant_{models[1]}_kappa']}", fontsize=7, transform=fig.transFigure, ha='center')


plt.subplots_adjust(bottom=0.1, top=0.9, left=0.095, right=0.998, hspace=0.5)

plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_5stages.svg'))
plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_5stages.eps'))
plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_5stages.tiff'), dpi=400)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

### same analysis but for (W, NREM, REM)

In [15]:
staging_cols = [x for x in data_concordant.columns if 'stage' in x]
# set all NREM stages (1, 2, 3) to 1:
data_concordant_3stages = data_concordant.copy().replace(2, 1)
data_concordant_3stages.replace(3, 1, inplace=True)
data_discordant_3stages = data_discordant.copy().replace(2, 1)
data_discordant_3stages.replace(3, 1, inplace=True)

labels_order_3stages = [5, 4, 1]
labels_names_3stages = ['W', 'R', 'NREM']

models = ['stage_pred_amendsumeffort', 'stage_pred_ecg_nn']

for model in models:

    print(model)
    
    cm_absolute = confusion_matrix(data_concordant_3stages['stage_pred_expert'], data_concordant_3stages[model], labels = labels_order_3stages)
    cm_normalized = confusion_matrix(data_concordant_3stages['stage_pred_expert'], data_concordant_3stages[model], labels = labels_order_3stages, normalize='true')
    kappa = cohen_kappa_score(data_concordant_3stages['stage_pred_expert'], data_concordant_3stages[model])
    
    cm_dict[f"concordant_{model}_absolute_3stages"] = cm_absolute
    cm_dict[f"concordant_{model}_normalized_3stages"] = cm_normalized
    cm_dict[f"concordant_{model}_kappa_3stages"] = np.round(kappa, 2)

    cm_absolute = confusion_matrix(data_discordant_3stages['stage_pred_expert'], data_discordant_3stages[model], labels = labels_order_3stages)
    cm_normalized = confusion_matrix(data_discordant_3stages['stage_pred_expert'], data_discordant_3stages[model], labels = labels_order_3stages, normalize='true')
    kappa = cohen_kappa_score(data_discordant_3stages['stage_pred_expert'], data_discordant_3stages[model])

    cm_dict[f"discordant_{model}_absolute_3stages"] = cm_absolute
    cm_dict[f"discordant_{model}_normalized_3stages"] = cm_normalized
    cm_dict[f"discordant_{model}_kappa_3stages"] = np.round(kappa, 2)

    
# print()
# print(confusion_matrix(data_concordant['stage_pred_expert'], data_concordant[model], normalize='true').round(2))

stage_pred_amendsumeffort
stage_pred_ecg_nn


In [16]:
cmap_ss5_absolute = LinearSegmentedColormap.from_list('ss5_absolute', ['white', 'black'], N=100)
model_labels = {'stage_pred_amendsumeffort': 'Breathing-based model',
               'stage_pred_ecg_nn': 'HRV-based model'}

In [17]:
plt.close('all')

fig, ax = plt.subplots(2, 4, figsize=(7.6, 5))

for i_model, model in enumerate(models):

    # CONCORDANT 
    i_axis = (0, 0 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"concordant_{model}_absolute_3stages"], display_labels=labels_names_3stages)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='d')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title(f"Confusion Matrix")
    ax[i_axis].set_ylabel(None)
    ax[i_axis].set_yticklabels(labels_names_3stages, rotation = 90, va='center')

    i_axis = (0, 1 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"concordant_{model}_normalized_3stages"], display_labels=labels_names_3stages)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis])
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('Confusion Matrix Normalized')
    ax[i_axis].set_ylabel(None)
    ax[i_axis].set_yticklabels(labels_names_3stages, rotation = 90, va='center')

    # DISCORDANT 
    i_axis = (1, 0 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"discordant_{model}_absolute_3stages"], display_labels=labels_names_3stages)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='d')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('Confusion Matrix')
    ax[i_axis].set_ylabel(None)
    ax[i_axis].set_yticklabels(labels_names_3stages, rotation = 90, va='center')

    i_axis = (1, 1 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"discordant_{model}_normalized_3stages"], display_labels=labels_names_3stages)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis])
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('Confusion Matrix Normalized')
    ax[i_axis].set_ylabel(None)
    ax[i_axis].set_yticklabels(labels_names_3stages, rotation = 90, va='center')
    
ax[(0, 0)].set_ylabel(f'Expert label')
ax[(1, 0)].set_ylabel(f'Expert label')
plt.text(0.31, 0.97, 'Breathing', transform=fig.transFigure, ha='center', fontsize=8, fontweight='bold')
plt.text(0.77, 0.97, 'HRV', transform=fig.transFigure, ha='center', fontsize=8, fontweight='bold')
plt.text(-0.4, 0.5, 'Concordant data', transform=ax[(0, 0)].transAxes, rotation=90, va='center', fontsize=8, fontweight='bold')
plt.text(-0.4, 0.5, 'Discordant data', transform=ax[(1, 0)].transAxes, rotation=90, va='center', fontsize=8, fontweight='bold')

# add Cohen's kappa
plt.text(0.31, 0.93, f"Cohen's kappa = {cm_dict[f'concordant_{models[0]}_kappa_3stages']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.77, 0.93, f"Cohen's kappa = {cm_dict[f'concordant_{models[1]}_kappa_3stages']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.31, 0.45, f"Cohen's kappa = {cm_dict[f'discordant_{models[0]}_kappa_3stages']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.77, 0.45, f"Cohen's kappa = {cm_dict[f'discordant_{models[1]}_kappa_3stages']}", fontsize=7, transform=fig.transFigure, ha='center')

plt.subplots_adjust(bottom=0.1, top=0.9, left=0.095, right=0.998, hspace=0.5)

plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_3stages.svg'))
plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_3stages.eps'))
plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_3stages.tiff'), dpi=400)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

### together 5 and 3 stages:

In [18]:
plt.close('all')

fig, ax = plt.subplots(4, 4, figsize=(7.6, 10.2))

for i_model, model in enumerate(models):

    # CONCORDANT 
    i_axis = (0, 0 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"concordant_{model}_absolute"], display_labels=labels_names)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='d')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title(f"Confusion matrix")
    ax[i_axis].set_ylabel(None)

    i_axis = (0, 1 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"concordant_{model}_normalized"], display_labels=labels_names)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis])
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('normalized')
    ax[i_axis].set_ylabel(None)

    # DISCORDANT 
    i_axis = (1, 0 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"discordant_{model}_absolute"], display_labels=labels_names)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='d')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('Confusion matrix')
    ax[i_axis].set_ylabel(None)

    i_axis = (1, 1 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"discordant_{model}_normalized"], display_labels=labels_names)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis])
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('normalized')
    ax[i_axis].set_ylabel(None)

    
ax[(0, 0)].set_ylabel(f'Expert label')
ax[(1, 0)].set_ylabel(f'Expert label')
plt.text(0.31, 1, 'Breathing', transform=fig.transFigure, ha='center', fontsize=8, fontweight='bold', va='top')
plt.text(0.77, 1, 'HRV', transform=fig.transFigure, ha='center', fontsize=8, fontweight='bold', va='top')
plt.text(-0.4, 0.5, 'Concordant data', transform=ax[(0, 0)].transAxes, rotation=90, va='center', fontsize=8, fontweight='bold')
plt.text(-0.4, 0.5, 'Discordant data', transform=ax[(1, 0)].transAxes, rotation=90, va='center', fontsize=8, fontweight='bold')

# add Cohen's kappa
plt.text(0.31, 0.967, f"Cohen's kappa = {cm_dict[f'concordant_{models[0]}_kappa']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.77, 0.967, f"Cohen's kappa = {cm_dict[f'concordant_{models[1]}_kappa']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.31, 0.72, f"Cohen's kappa = {cm_dict[f'discordant_{models[0]}_kappa']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.77, 0.72, f"Cohen's kappa = {cm_dict[f'discordant_{models[1]}_kappa']}", fontsize=7, transform=fig.transFigure, ha='center')


for i_model, model in enumerate(models):

    # CONCORDANT 
    i_axis = (2, 0 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"concordant_{model}_absolute_3stages"], display_labels=labels_names_3stages)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='d')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title(f"Confusion matrix")
    ax[i_axis].set_ylabel(None)
    ax[i_axis].set_yticklabels(labels_names_3stages, rotation = 90, va='center')

    i_axis = (2, 1 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"concordant_{model}_normalized_3stages"], display_labels=labels_names_3stages)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis])
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('normalized')
    ax[i_axis].set_ylabel(None)
    ax[i_axis].set_yticklabels(labels_names_3stages, rotation = 90, va='center')

    # DISCORDANT 
    i_axis = (3, 0 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"discordant_{model}_absolute_3stages"], display_labels=labels_names_3stages)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='d')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('Confusion matrix')
    ax[i_axis].set_ylabel(None)
    ax[i_axis].set_yticklabels(labels_names_3stages, rotation = 90, va='center')

    i_axis = (3, 1 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"discordant_{model}_normalized_3stages"], display_labels=labels_names_3stages)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis])
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
    ax[i_axis].set_title('normalized')
    ax[i_axis].set_ylabel(None)
    ax[i_axis].set_yticklabels(labels_names_3stages, rotation = 90, va='center')
    
ax[(2, 0)].set_ylabel(f'Expert label')
ax[(3, 0)].set_ylabel(f'Expert label')
# plt.text(0.31, 0.97, 'Breathing', transform=fig.transFigure, ha='center', fontsize=8, fontweight='bold')
# plt.text(0.77, 0.97, 'HRV', transform=fig.transFigure, ha='center', fontsize=8, fontweight='bold')
plt.text(-0.4, 0.5, 'Concordant data', transform=ax[(2, 0)].transAxes, rotation=90, va='center', fontsize=8, fontweight='bold')
plt.text(-0.4, 0.5, 'Discordant data', transform=ax[(3, 0)].transAxes, rotation=90, va='center', fontsize=8, fontweight='bold')

# add Cohen's kappa
plt.text(0.31, 0.47, f"Cohen's kappa = {cm_dict[f'concordant_{models[0]}_kappa_3stages']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.77, 0.47, f"Cohen's kappa = {cm_dict[f'concordant_{models[1]}_kappa_3stages']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.31, 0.223, f"Cohen's kappa = {cm_dict[f'discordant_{models[0]}_kappa_3stages']}", fontsize=7, transform=fig.transFigure, ha='center')
plt.text(0.77, 0.223, f"Cohen's kappa = {cm_dict[f'discordant_{models[1]}_kappa_3stages']}", fontsize=7, transform=fig.transFigure, ha='center')

plt.text(-0.48, 1.25,  '5 stages:', transform=ax[(0, 0)].transAxes, va='center', fontsize=8, fontweight='bold', fontstyle='italic')
plt.text(-0.48, 1.25, 'W-R-NREM:', transform=ax[(2, 0)].transAxes, va='center', fontsize=8, fontweight='bold', fontstyle='italic')

plt.subplots_adjust(bottom=0.04, top=0.96, left=0.095, right=0.998, hspace=0.4)

plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_3_and_5stages.svg'))
plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_3_and_5stages.eps'))
plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_3_and_5stages.tiff'), dpi=400)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [72]:
plt.close('all')

fig, ax = plt.subplots(2, 4, figsize=(7.6, 4.5))

for i_model, model in enumerate(models):

    # CONCORDANT 
    i_axis = (0, 0 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"concordant_{model}_normalized"], display_labels=labels_names)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='.2f')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
#     ax[i_axis].set_title(f"5 stages")
    ax[i_axis].set_ylabel(None)

    i_axis = (0, 1 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"concordant_{model}_normalized_3stages"], display_labels=labels_names_3stages)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='.2f')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
#     ax[i_axis].set_title('W-R-NREM')
    ax[i_axis].set_ylabel(None)
    ax[i_axis].set_yticklabels(labels_names_3stages, rotation = 90, va='center')

    # DISCORDANT 
    i_axis = (1, 0 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"discordant_{model}_normalized"], display_labels=labels_names)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='.2f')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
#     ax[i_axis].set_title(f"5 stages")
    ax[i_axis].set_ylabel(None)

    i_axis = (1, 1 + 2*i_model)
    disp = ConfusionMatrixDisplay(cm_dict[f"discordant_{model}_normalized_3stages"], display_labels=labels_names_3stages)
    disp.plot(colorbar=False,  cmap = cmap_ss5_absolute, ax=ax[i_axis], values_format='.2f')
    ax[i_axis].set_xlabel(f'Predicted sleep stage by\n{model_labels[model]}')
#     ax[i_axis].set_title('W-R-NREM')
    ax[i_axis].set_ylabel(None)
    ax[i_axis].set_yticklabels(labels_names_3stages, rotation = 90, va='center')

    
ax[(0, 0)].set_ylabel(f'Expert label')
ax[(1, 0)].set_ylabel(f'Expert label')
plt.text(0.31, 0.998, 'Breathing', transform=fig.transFigure, ha='center', fontsize=8, fontweight='bold', va='top')
plt.text(0.77, 0.998, 'HRV', transform=fig.transFigure, ha='center', fontsize=8, fontweight='bold', va='top')
plt.text(-0.65, 0.5, 'Concordant data', transform=ax[(0, 0)].transAxes, rotation=90, va='center', fontsize=8, fontweight='bold')
plt.text(-0.65, 0.5, 'Discordant data', transform=ax[(1, 0)].transAxes, rotation=90, va='center', fontsize=8, fontweight='bold')

# add Cohen's kappa
cohens_ypos = 1.02
plt.text(0.5, cohens_ypos, f"5 stages\nCohen's kappa = {cm_dict[f'concordant_{models[0]}_kappa']}", fontsize=7, transform=ax[0, 0].transAxes, ha='center', va='bottom')
plt.text(0.5, cohens_ypos, f"W-R-NREM\nCohen's kappa = {cm_dict[f'concordant_{models[0]}_kappa_3stages']}", fontsize=7, transform=ax[0, 1].transAxes, ha='center', va='bottom')
plt.text(0.5, cohens_ypos, f"5 stages\nCohen's kappa = {cm_dict[f'concordant_{models[1]}_kappa']}", fontsize=7, transform=ax[0, 2].transAxes, ha='center', va='bottom')
plt.text(0.5, cohens_ypos, f"W-R-NREM\nCohen's kappa = {cm_dict[f'concordant_{models[1]}_kappa_3stages']}", fontsize=7, transform=ax[0, 3].transAxes, ha='center', va='bottom')

plt.text(0.5, cohens_ypos, f"5 stages\nCohen's kappa = {cm_dict[f'discordant_{models[0]}_kappa']}", fontsize=7, transform=ax[1, 0].transAxes, ha='center', va='bottom')
plt.text(0.5, cohens_ypos, f"W-R-NREM\nCohen's kappa = {cm_dict[f'discordant_{models[0]}_kappa_3stages']}", fontsize=7, transform=ax[1, 1].transAxes, ha='center', va='bottom')
plt.text(0.5, cohens_ypos, f"5 stages\nCohen's kappa = {cm_dict[f'discordant_{models[1]}_kappa']}", fontsize=7, transform=ax[1, 2].transAxes, ha='center', va='bottom')
plt.text(0.5, cohens_ypos, f"W-R-NREM\nCohen's kappa = {cm_dict[f'discordant_{models[1]}_kappa_3stages']}", fontsize=7, transform=ax[1, 3].transAxes, ha='center', va='bottom')

n_concordant = cm_dict[f'concordant_{models[0]}_absolute'].sum(axis=1)
n_discordant = cm_dict[f'discordant_{models[0]}_absolute'].sum(axis=1)
ax[0, 0].set_yticklabels(labels_names[i] + f'\n(N={n_concordant[i]:,})' for i in range(5))
ax[1, 0].set_yticklabels(labels_names[i] + f'\n(N={n_discordant[i]:,})' for i in range(5))

plt.subplots_adjust(bottom=0, top=1, left=0.12, right=0.998, hspace=0.0, wspace=0.3)

plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_normalized_3_and_5stages.svg'))
plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_normalized_3_and_5stages.eps'))
plt.savefig(os.path.join(plots_savedir, 'confusion_matrices_normalized_3_and_5stages.tiff'), dpi=400)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

array([29879, 11514, 12761, 63210, 13793])

In [20]:
model

'stage_pred_ecg_nn'

In [21]:
labels_names_3stages

['W', 'R', 'NREM']

In [22]:
data_concordant['expert_agreed'] = (data_concordant.stage_pred_expert.values[:, np.newaxis] == data_concordant[['stage_pred_ecg_nn', 'stage_pred_amendsumeffort']].values).any(axis=1)
data_discordant['expert_agreed'] = (data_discordant.stage_pred_expert.values[:, np.newaxis] == data_discordant[['stage_pred_ecg_nn', 'stage_pred_amendsumeffort']].values).any(axis=1)

In [23]:
data_concordant['expert_agreed'].mean()

0.7319700816578604

In [24]:
data_discordant['expert_agreed'].mean()

0.6412190565156469

In [25]:
data_concordant_3stages['expert_agreed'] = (data_concordant_3stages.stage_pred_expert.values[:, np.newaxis] == data_concordant_3stages[['stage_pred_ecg_nn', 'stage_pred_amendsumeffort']].values).any(axis=1)
data_discordant_3stages['expert_agreed'] = (data_discordant_3stages.stage_pred_expert.values[:, np.newaxis] == data_discordant_3stages[['stage_pred_ecg_nn', 'stage_pred_amendsumeffort']].values).any(axis=1)

In [26]:
data_concordant_3stages['expert_agreed'].mean()

0.8947978377059554

In [27]:
data_discordant_3stages['expert_agreed'].mean()

0.9068192433442317